## Setup

In [ ]:
# set libraries to refresh
%load_ext autoreload
%autoreload 2

In [41]:
from bs4 import BeautifulSoup

In [42]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import shapely
import contextily as ctx
import xyzservices as xyz
import matplotlib.pyplot as plt

# import kml reading and set supported driver
import fiona
fiona.drvsupport.supported_drivers["KML"] = "rw"

In [43]:
from gridsample.utils import create_ids, save_shapefiles

In [44]:
ROOT_DIR = Path("../")
DATA_DIR = ROOT_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "00_raw"
PROCESSED_DATA_DIR = DATA_DIR / "01_processed" / "Solar Parks"

In [51]:
# ### Unzip KMZ to get KML - USE GOOGLE EARTH TO DO THIS CORRECTLY
# import zipfile

# # Extract the .kml file from the .kmz file
# with zipfile.ZipFile(RAW_DATA_DIR / "solar_park_shapefiles" / "dhar181224.kmz", 'r') as kmz:
#     kmz.extractall(RAW_DATA_DIR / "solar_park_shapefiles" / "dhar181224" )

### Read KML and process columns

In [70]:
gdfs = []

for filename in ["sagar_khamkuwa", "sagar_mokalpur", "sagar_tekapar"]:
    gdf = gpd.read_file(
        RAW_DATA_DIR / "solar_park_shapefiles" / "Google Earth" / f"{filename}.kml",
        driver="KML"
    )
    gdf["source"] = filename
    gdfs.append(gdf)

raw_gdf = pd.concat(gdfs, ignore_index=True)

In [71]:
raw_gdf.geometry = raw_gdf.geometry.apply(lambda x: shapely.wkb.loads(shapely.wkb.dumps(x, output_dimension=2)))

In [ ]:
raw_gdf.plot(column="source", legend=True)

In [74]:
# raw_gdf = gpd.read_file(
#     RAW_DATA_DIR / "solar_park_shapefiles" / "dhar181224" / "doc.kml",
#     driver="KML"
# )
# # remove z-dimension
# raw_gdf.geometry = raw_gdf.geometry.apply(lambda x: shapely.wkb.loads(shapely.wkb.dumps(x, output_dimension=2)))

# save_shapefiles(raw_gdf[:2000], PROCESSED_DATA_DIR, "dhar181224_processed_first2k", formats=["kml"])

#### Parse description variables

In [75]:
def description_parser(html_content):
    # Parse the HTML content
    soup = BeautifulSoup(html_content, 'html.parser')

    # Find the inner table containing the attributes
    inner_table = soup.find_all('table')[1]

    # Extract rows from the inner table
    rows = inner_table.find_all('tr')

    # Create a dictionary to store attributes and their values
    data = {}
    for row in rows:
        cols = row.find_all('td')
        if len(cols) == 2:
            key = cols[0].text.strip()
            value = cols[1].text.strip()
            data[key] = value

    return pd.DataFrame([data])

In [76]:
# make dataframe of variables
data = [description_parser(raw_gdf["Description"][i]) for i in range(len(raw_gdf))]
df = pd.concat(data)
df.set_index(raw_gdf.index, inplace=True)

# merge with shapes
raw_gdf.drop(columns=["Name", "Description"], inplace=True)
gdf = raw_gdf.merge(df, left_index=True, right_index=True)

In [ ]:
gdf

In [ ]:
gdf.plot(column="PAR_TYPE")

In [80]:
save_shapefiles(gdf, PROCESSED_DATA_DIR, "sagar220724_processed_all", formats=["kml"])

In [ ]:
# get unique values for all columns
for col in gdf.columns:
    print(f"Column: {col}")
    print(gdf[col].unique())
    print("\n")